# ARC-AGI-3 Kaggle Submission Notebook — Offline Version

Notebook adaptado para Kaggle sin internet, usando `OperationMode.OFFLINE` y los `environment_files` incluidos en la competición.

Esta versión evita llamadas a `three.arcprize.org`, detecta entornos automáticamente y ejecuta los juegos locales directamente.

## 1. Instalación de dependencias

In [ ]:
!pip install -q "arc-agi" "pillow<12"

## 2. Imports

In [ ]:
import random
from collections import defaultdict

import arc_agi
from arc_agi.base import OperationMode


## 3. Configuración
Ajustá estos parámetros según la experimentación que quieras hacer en Kaggle.

In [ ]:
ENV_DIR = "/kaggle/input/competitions/arc-prize-2026-arc-agi-3/environment_files"
MAX_STEPS = 60
MAX_GAMES = None
RANDOM_SEED = 42
random.seed(RANDOM_SEED)

## 4. Instanciación offline de Arcade
Esta es la parte crítica para Kaggle: `OFFLINE` + `environment_files` locales.

In [ ]:
arc = arc_agi.Arcade(
    operation_mode=OperationMode.OFFLINE,
    environments_dir=ENV_DIR
)

print("Arcade offline creada OK")
print("Entornos detectados:", len(arc.available_environments))

## 5. Detección automática de juegos
Se derivan los prefijos de juego desde `arc.available_environments` en vez de hardcodear una lista manual.

In [ ]:
detected_game_ids = []
seen_prefixes = set()

for env_info in arc.available_environments:
    full_id = env_info.game_id
    prefix = full_id.split("-")[0]
    if prefix not in seen_prefixes:
        detected_game_ids.append(prefix)
        seen_prefixes.add(prefix)

GAME_IDS = detected_game_ids[:MAX_GAMES] if MAX_GAMES is not None else detected_game_ids

print("Total detected game prefixes:", len(GAME_IDS))
print("Primeros 20:", GAME_IDS[:20])

## 6. Agente versión 2
Agente base con memoria simple, penalización de repetición y exploración espacial sistemática.

In [ ]:
class SubmissionAgentV2:
    def __init__(self, max_steps=60):
        self.max_steps = max_steps
        self.coord_candidates = [
            (0, 0), (0, 15), (15, 0), (15, 15),
            (3, 3), (7, 7), (8, 4), (4, 8), (10, 10)
        ]
        self.reset()

    def reset(self):
        self.visited_states = defaultdict(int)
        self.action_counts = defaultdict(int)
        self.action_success = defaultdict(float)
        self.last_reward = 0.0
        self.coord_index = 0

    def state_key(self, obs):
        return repr(obs)[:1000]

    def choose_coord(self):
        coord = self.coord_candidates[self.coord_index % len(self.coord_candidates)]
        self.coord_index += 1
        return coord

    def act(self, env, obs):
        if obs is not None:
            self.visited_states[self.state_key(obs)] += 1

        actions = list(env.action_space)

        def score(action):
            name = str(action)
            return self.action_success[name] - 0.30 * self.action_counts[name]

        actions.sort(key=score, reverse=True)
        pool = actions[:max(1, len(actions)//2)]
        action = random.choice(pool)

        action_data = {}
        if hasattr(action, "is_complex") and action.is_complex():
            x, y = self.choose_coord()
            action_data = {"x": x, "y": y}

        self.action_counts[str(action)] += 1
        return action, action_data

    def update(self, action, reward):
        delta = reward - self.last_reward
        self.action_success[str(action)] += delta
        self.last_reward = reward


## 7. Normalización de `env.step`

In [ ]:
def parse_step_result(step_result):
    if not isinstance(step_result, tuple):
        return step_result, 0.0, False, False, {}

    if len(step_result) >= 5:
        return (
            step_result[0],
            step_result[1],
            bool(step_result[2]),
            bool(step_result[3]),
            step_result[4] if isinstance(step_result[4], dict) else {}
        )

    if len(step_result) == 4:
        obs, reward, done, info = step_result
        return obs, reward, bool(done), False, info if isinstance(info, dict) else {}

    if len(step_result) == 3:
        obs, reward, done = step_result
        return obs, reward, bool(done), False, {}

    if len(step_result) == 2:
        obs, reward = step_result
        return obs, reward, False, False, {}

    return step_result[0], 0.0, False, False, {}


## 8. Runner por juego (offline)

In [ ]:
def run_single_game_offline(game_id, max_steps=60):
    env = arc.make(game_id, render_mode=None)
    agent = SubmissionAgentV2(max_steps=max_steps)

    obs = None
    reward = 0.0
    terminated = False
    truncated = False
    info = {}
    trace = []

    for step in range(agent.max_steps):
        action, action_data = agent.act(env, obs)
        step_result = env.step(action, data=action_data)
        obs, reward, terminated, truncated, info = parse_step_result(step_result)
        agent.update(action, reward)

        trace.append({
            "step": step,
            "action": str(action),
            "action_data": action_data,
            "reward": reward,
            "terminated": terminated,
            "truncated": truncated,
            "info": info
        })

        if terminated or truncated:
            break

    return {
        "game_id": game_id,
        "steps": len(trace),
        "final_reward": reward,
        "scorecard": str(arc.get_scorecard())
    }


## 9. Runner batch
Ejecuta varios juegos offline y resume resultados.

In [ ]:
def run_batch(game_ids, max_steps=60):
    results = []
    for game_id in game_ids:
        try:
            result = run_single_game_offline(game_id, max_steps=max_steps)
            results.append(result)
            print(f"OK  {game_id} | steps={result['steps']} | final_reward={result['final_reward']}")
        except Exception as e:
            results.append({
                "game_id": game_id,
                "error": str(e)
            })
            print(f"ERR {game_id} | {e}")
    return results


## 10. Ejecución

In [ ]:
batch_results = run_batch(GAME_IDS, max_steps=MAX_STEPS)
batch_results[:5]

## 11. Resumen agregado

In [ ]:
successful = [r for r in batch_results if 'error' not in r]
failed = [r for r in batch_results if 'error' in r]

summary = {
    "games_requested": len(GAME_IDS),
    "games_succeeded": len(successful),
    "games_failed": len(failed),
    "avg_steps": sum(r['steps'] for r in successful) / len(successful) if successful else None,
    "avg_final_reward": sum(r['final_reward'] for r in successful) / len(successful) if successful else None
}

summary

## 12. Nota final
Esta versión ya está alineada con Kaggle offline: evita API online, usa `environment_files` locales y detecta automáticamente los juegos disponibles.